# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [26]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, ttest_1samp, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [4]:
# Run this code:
salaries = pd.read_csv('../data/Current_Employee_Names__Salaries__and_Position_Titles.csv')

Examine the `salaries` dataset using the `head` function below.

In [12]:
# Your code here
salaries.head(10)

,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"AARON, JEFFERY M",SERGEANT,POLICE,F,Salary,NaN,101442.0,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),POLICE,F,Salary,NaN,94122.0,NaN
2,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,F,Salary,NaN,101592.0,NaN
3,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,F,Salary,NaN,110064.0,NaN
4,"ABASCAL, REECE E",TRAFFIC CONTROL AIDE-HOURLY,OEMC,P,Hourly,20.0,NaN,19.86
5,"ABBASI, CHRISTOPHER",STAFF ASST TO THE ALDERMAN,CITY COUNCIL,F,Salary,NaN,50436.0,NaN
6,"ABBATACOLA, ROBERT J",ELECTRICAL MECHANIC,AVIATION,F,Hourly,40.0,NaN,46.10
7,"ABBATE, JOSEPH L",POOL MOTOR TRUCK DRIVER,STREETS & SAN,F,Hourly,40.0,NaN,35.60
8,"ABBATEMARCO, JAMES J",FIRE ENGINEER-EMT,FIRE,F,Salary,NaN,103350.0,NaN
9,"ABBATE, TERRY M",POLICE OFFICER,POLICE,F,Salary,NaN,93354.0,NaN


# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Import the correct one sample test function from scipy and perform the hypothesis test for a 95% two sided confidence interval.

In [31]:
import scipy.stats as stats

In [34]:
# Test hourly wages against $30/hr using a two-sided one-sample t-test

hourly_mask = salaries["Salary or Hourly"] == "Hourly"
hourly_rates = salaries.loc[hourly_mask, "Hourly Rate"].dropna()

result = stats.ttest_1samp(
    hourly_rates,
    popmean=30,
    alternative="two-sided"
)
ci = result.confidence_interval(confidence_level=0.95)

print(f"Sample mean: ${hourly_rates.mean():.2f}/hr")
print(f"t-statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.4g}")
print(f"95% CI: (${ci.low:.2f}, ${ci.high:.2f})")

if result.pvalue < 0.05:
    print("Reject H0: the mean hourly wage differs from $30/hr.")
else:
    print("Fail to reject H0.")

Sample mean: $32.79/hr
t-statistic: 20.6198
p-value: 4.323e-92
95% CI: ($32.52, $33.05)
Reject H0: the mean hourly wage differs from $30/hr.


# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives us information about the true mean of the population. So for a 95% confidence interval, we are 95% sure that the mean of the population is within the confidence interval. 
).

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter and the standard error for the scale. The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [35]:
# Your code here
from scipy.stats import t, sem

hourly_mask = salaries["Salary or Hourly"] == "Hourly"
hourly_rates = salaries.loc[hourly_mask, "Hourly Rate"].dropna()

confidence_interval = t.interval(
    0.95,
    df=len(hourly_rates) - 1,
    loc=hourly_rates.mean(),
    scale=sem(hourly_rates)
)

print(f"95% confidence interval: ${confidence_interval[0]:.2f} to ${confidence_interval[1]:.2f} per hour")

95% confidence interval: $32.52 to $33.05 per hour


# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the number of hourly workers in the City of Chicago is significantly different from 25% at the 95% confidence level.

In [37]:
# Your code here
from statsmodels.stats.proportion import proportions_ztest

hourly_mask = salaries["Salary or Hourly"] == "Hourly"
count = hourly_mask.sum()
nobs = len(salaries)

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    value=0.25,
    prop_var=0.25,
    alternative="two-sided"
)

print(f"Hourly worker proportion: {count / nobs:.2%}")
print(f"z-statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4g}")

if p_value < 0.05:
    print("Reject H0: the hourly-worker proportion differs from 25%.")
else:
    print("Fail to reject H0: insufficient evidence that it differs from 25%.")

Hourly worker proportion: 24.18%
z-statistic: -3.4705
p-value: 0.0005194
Reject H0: the hourly-worker proportion differs from 25%.
